Para começar:

In [64]:
%pip install requests pandas
import requests
import base64
import urllib.parse
import pandas as pd
import numpy as np
import time
import re
import unicodedata

Note: you may need to restart the kernel to use updated packages.


=============================================================================
### Pipeline de extração e curadoria de patentes do EPO (siRNA: 2026)
=============================================================================

Este script automatiza a extração e curadoria de metadados bibliográficos de 
patentes utilizando a API Open Patent Services (OPS) do European Patent Office (EPO). 
Os resultados extraídos são consolidados e exportados num formato tabular (CSV) 
otimizado para análise estatística e visualização em R.

A pesquisa (Query CQL) foi rigorosamente desenhada para maximizar o recall de 
inovações em terapias genéticas baseadas em siRNA e RNAi para aplicação estritamente 
humana, garantindo a integridade dos dados através de uma lógica de triagem avançada.

### Principais Funcionalidades:
---------------------------
1. Pesquisa Estruturada de Metadados: 
   As patentes apenas são retidas se os termos-alvo constarem em secções 
   críticas (Título, Resumo ou Reivindicações) em conjunção com as classes 
   CPC adequadas (Biotecnologia/Farmacologia).

2. Gestão Dinâmica de Sessão (OAuth 2.0): 
   Integração de um temporizador interno que previne timeouts em extrações 
   de grande volume, renovando o token de acesso proativamente antes do 
   limite de 20 minutos imposto pelo EPO.

3. Filtragem Off-Target (Escudo Anti-Veterinária/Agropecuária): 
   Filtro lexical negativo para retirar patentes direcionadas a pecuária, 
   aquacultura, botânica e entomologia, retendo simultaneamente a permissão 
   para modelos animais de laboratório (ex: ratinhos transgénicos).

4. Anotação Automática (Tagging): 
   Manutenção propositada de tecnologias como CRISPR, miRNA, circRNA, ASOs 
   para evitar perda de dados por sistemas de entrega cruzados, sinalizando-as na 
   coluna 'Aviso_Tecnologia' para curadoria técnica manual subsequente.

5. Estratificação por Applicant (Requerente): 
   Parâmetro configurável para restringir as extrações a patentes detidas 
   por uma entidade específica (Competitive Landscaping).

6. Monitorização de Quota da API: 
   Geração de relatório automatizado no final da iteração (Audit), documentando 
   o consumo e garantindo o cumprimento da política de Fair Use 
   semanal do EPO.

=============================================================================

In [ ]:
# ==========================================
# 1. CONFIGURACOES GERAIS E CREDENCIAIS
# ==========================================
# Credenciais da API Open Patent Services (OPS) do European Patent Office (EPO)
# ATENÇÃO: Substitui pelas tuas novas credenciais geradas no portal do EPO.
CONSUMER_KEY = "" # guardado por privacidade
CONSUMER_SECRET = "" # guardado por privacidade

# Dicionario de pesquisa principal: Focado na identificacao de tecnologias de siRNA e RNAi.
termos_rna = [
    "siRNA*", "SiRNA*", "iRNA*", "IRNA*",
    "RNAi*", "RNAI*",
    "small interfer* RNA*", "short interfer* RNA*",
    "RNA interfer*",
    "RNA interfer* agent*", "RNAi agent*",
    "double-stranded ribonucleic acid*", "double stranded ribonucleic acid*",
    "double stranded RNA*", "double stranded RNAi*",
    "dsRNA*", "dsRNAi*",
    "si-RNA*", "ds-RNA*",
    "short-interfer* RNA*", "small-interfer* RNA*",
    "RNAi molecule*", "RNA interfer* molecule*",
    "siRNA agent*", "siRNA molecule*",
    "silenc* RNA*",
    "ds-oligonucleotide*", "double stranded oligonucleotide*", "RNAi oligonucleotide*"
]

# Termos de alerta: Utilizados em pos-processamento para assinalar possiveis tecnologias concorrentes.
termos_concorrentes = 'CRISPR|Cas9|Cas13|antisense|ASO|microRNA|miRNA|circRNA|circular RNA|aptamer'

# Filtro de exclusao (Anti-Agricultura/Veterinaria): Dicionario focado em nomes comuns e cientificos
# para eliminar patentes fora do ambito da saude humana.
termos_vet_agri = 'porcine|bovine|equine|swine|poultry|livestock|aquaculture|fish|cynoglossus|porc|poisson|veterinary|veterinaire|canine|feline|cattle|sheep|chicken|pig|whitefly|pest|insect|aphid|nematode|planthopper|bee|bees|moustique|macrobrachium|prawn|shrimp|plutella|bactrocera|paederus|hühnern|vogelgrippe|chilo suppressalis|bemisia tabaci'

# Criterio de Classificacao de Patentes (CPC): Limita a pesquisa estritamente a biotecnologia e farmacologia.
cpc_query = '(cpc="C12N15/113" OR cpc="A61K31/713")'

# ------------------------------------------
# PERIODO TEMPORAL DE PESQUISA
# ------------------------------------------
# A pesquisa pode ser definida para um unico ano ou um intervalo de anos (Publication Date).
anos_para_pesquisar = [2026]

# ------------------------------------------
# FILTRO DE APPLICANT (EMPRESA/INSTITUICAO)
# ------------------------------------------
# Permite restringir a pesquisa a um requerente especifico para fins de analise da concorrencia.
FILTRAR_POR_APPLICANT = False       # Alterar para True para ativar a restricao
NOME_APPLICANT = "Alnylam"          # Nome do requerente a pesquisar (ex: "Alnylam")


# ==========================================
# 2. GESTOR DE TOKEN E FUNCOES AUXILIARES
# ==========================================

# Variaveis globais de estado para gerir a validade da sessao com a API do EPO
TOKEN_ATUAL = None
HORA_GERACAO_TOKEN = 0

def obter_token_valido():
    """
    Gere a autenticacao OAuth 2.0 com a API do EPO.
    O token tem uma validade estrita de 20 minutos. Esta funcao implementa uma margem
    de seguranca de 15 minutos (900 segundos) para forcar a renovacao antes da expiracao,
    evitando quebras durante extracoes de longa duracao.
    """
    global TOKEN_ATUAL, HORA_GERACAO_TOKEN
    agora = time.time()
    
    if TOKEN_ATUAL is None or (agora - HORA_GERACAO_TOKEN) > 900:
        print("\n[SEGURANCA] A gerar um novo Token de Acesso EPO...")
        url = "https://ops.epo.org/3.2/auth/accesstoken"
        auth = base64.b64encode(f"{CONSUMER_KEY}:{CONSUMER_SECRET}".encode()).decode()
        res = requests.post(url,
            headers={'Authorization': f'Basic {auth}', 'Content-Type': 'application/x-www-form-urlencoded'},
            data={'grant_type': 'client_credentials'})
        res.raise_for_status()
        
        TOKEN_ATUAL = res.json()['access_token']
        HORA_GERACAO_TOKEN = time.time()
        
    return TOKEN_ATUAL

def clean_val(node):
    """
    Funcao utilitaria para normalizar os dados extraidos da arvore JSON complexa do EPO.
    Garante que dicionarios com o formato {'$': 'valor'} sao convertidos para strings limpas.
    """
    if isinstance(node, dict):
        return node.get('$', '')
    return str(node) if node else ""

def pesquisar_ids(cql_query, ano):
    """
    Executa a pesquisa bibliografica utilizando a linguagem CQL do EPO.
    A API limita o retorno a um maximo de 2000 resultados por query, logo, a extraccao
    e paginada em blocos de 100 registos ('X-OPS-Range').
    """
    q = urllib.parse.quote(f'{cql_query} AND pd={ano}')
    ids_extraidos = []
    
    start = 1
    while start <= 2000:
        token = obter_token_valido()
        end = start + 99
        headers = {
            'Authorization': f'Bearer {token}',
            'Accept': 'application/json',
            'X-OPS-Range': f'{start}-{end}'
        }
        res = requests.get(f"https://ops.epo.org/3.2/rest-services/published-data/search?q={q}", headers=headers)

        if res.status_code != 200:
            if res.status_code == 404:
                # O codigo 404 indica que nao ha mais resultados para as paginas seguintes.
                break
            else:
                print(f"\n[ERRO DA API EPO] O servidor rejeitou o pedido. Codigo: {res.status_code}")
                print(f"Detalhes: {res.text}")
                break

        data = res.json().get('ops:world-patent-data', {}).get('ops:biblio-search', {})
        docs = data.get('ops:search-result', {}).get('ops:publication-reference', [])
        
        if not docs:
            break
        if isinstance(docs, dict):
            docs = [docs]

        # Iterar sobre os documentos e extrair o formato 'docdb' para unificacao do ID.
        for doc in docs:
            dids = doc.get('document-id', doc.get('ops:document-id', []))
            if isinstance(dids, dict):
                dids = [dids]
            
            docdb_node = next((d for d in dids if d.get('@document-id-type') == 'docdb'), None)
            if docdb_node:
                c = clean_val(docdb_node.get('country', ''))
                num = clean_val(docdb_node.get('doc-number', ''))
                k = clean_val(docdb_node.get('kind', ''))
                id_parts = [p for p in [c, num, k] if p]
                if len(id_parts) >= 2:
                    ids_extraidos.append(".".join(id_parts))

        start += 100
        time.sleep(2)  # Respeito absoluto pelo limite de requisicoes (throttle) do EPO.
        
    return ids_extraidos

def obter_detalhes(lista_ids):
    """
    Processamento em batch para a obtencao de dados bibliograficos detalhados.
    Para minimizar chamadas de rede e consumo de quota, agregam-se ate 50 IDs por pedido.
    Se o lote falhar, existe um mecanismo de seguranca que processa os IDs individualmente.
    """
    print(f"\n[INFO] A iniciar extracao de metadados para {len(lista_ids)} patentes...")
    resultados = []
    for i in range(0, len(lista_ids), 50):
        token = obter_token_valido()
        lote = lista_ids[i:i+50]
        url = f"https://ops.epo.org/3.2/rest-services/published-data/publication/docdb/{','.join(lote)}/biblio"
        res = requests.get(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'})

        if res.status_code == 200:
            _extrair_doc(res.json(), resultados)
        elif res.status_code == 404:
            # Fallback para falha parcial no lote
            for id_individual in lote:
                url_ind = f"https://ops.epo.org/3.2/rest-services/published-data/publication/docdb/{id_individual}/biblio"
                r = requests.get(url_ind, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'})
                if r.status_code == 200:
                    _extrair_doc(r.json(), resultados)
                time.sleep(0.5)
        time.sleep(3)
    return resultados

def _extrair_doc(json_data, resultados):
    """
    Funcao de parsing que navega na arvore JSON recebida pela API do EPO.
    Extrai metadados essenciais: Titulo, Datas, Requerente, Identificadores e CPCs.
    Aplica prioridade ao idioma Ingles em todos os campos de texto.
    """
    data = json_data.get('ops:world-patent-data', {})
    ex_docs = data.get('ops:exchange-documents') or data.get('exchange-documents') or {}
    docs = ex_docs.get('ops:exchange-document') or ex_docs.get('exchange-document') or []
    
    if isinstance(docs, dict):
        docs = [docs]

    for doc in docs:
        bib = doc.get('ops:bibliographic-data', doc.get('bibliographic-data', {}))
        
        # 1. Tratamento da extracao do Titulo (Prioridade EN)
        t_node = bib.get('ops:invention-title', bib.get('invention-title', []))
        if isinstance(t_node, dict):
            t_node = [t_node]
            
        titulo = "Sem titulo"
        if t_node:
            titulo_en = None
            for t in t_node:
                if isinstance(t, dict) and t.get('@lang', '').lower() == 'en':
                    titulo_en = clean_val(t)
                    break
            titulo = titulo_en if titulo_en else clean_val(t_node[0])

        # 2. Extracao de Requerentes (Applicants) com Prioridade Ingles
        applicants_node = bib.get('parties', {}).get('applicants', {}).get('applicant', [])
        if isinstance(applicants_node, dict):
            applicants_node = [applicants_node]
        
        applicants_lista = []
        for app in applicants_node:
            nomes_disponiveis = app.get('applicant-name', {}).get('name', [])
            if isinstance(nomes_disponiveis, str) or isinstance(nomes_disponiveis, dict):
                nomes_disponiveis = [nomes_disponiveis]
                
            nome_final = ""
            nome_en = None
            
            # Procurar se existe alguma versao do nome explicitamente em ingles
            for n in nomes_disponiveis:
                if isinstance(n, dict) and n.get('@lang', '').lower() == 'en':
                    nome_en = clean_val(n)
                    break
                    
            if nome_en:
                nome_final = nome_en
            elif len(nomes_disponiveis) > 0:
                nome_final = clean_val(nomes_disponiveis[0])
                
            if nome_final:
                applicants_lista.append(nome_final)
                
        requerentes_str = " | ".join(applicants_lista)

        # 3. Tratamento da extracao dos Referenciais (ID)
        p_ref = bib.get('ops:publication-reference', bib.get('publication-reference', {}))
        d_ids = p_ref.get('ops:document-id', p_ref.get('document-id', []))
        if isinstance(d_ids, dict):
            d_ids = [d_ids]
        
        dt, cc, nr, kc = "", "", "", ""
        for d in d_ids:
            if not dt:
                dt = clean_val(d.get('date', ''))
            if d.get('@document-id-type') == 'docdb':
                cc = clean_val(d.get('country', '')).strip()
                nr = clean_val(d.get('doc-number', '')).replace(".", "").replace("-", "").strip()
                kc = clean_val(d.get('kind', '')).strip()

        # 4. Tratamento e consolidacao de Codigos CPC
        cpcs_set = set()
        c_wrap = bib.get('ops:patent-classifications', bib.get('patent-classifications', {}))
        c_list = c_wrap.get('ops:patent-classification', c_wrap.get('patent-classification', []))
        if isinstance(c_list, dict):
            c_list = [c_list]
        
        for c in c_list:
            s = clean_val(c.get('section',''))
            cl = clean_val(c.get('class',''))
            sub = clean_val(c.get('subclass',''))
            main = clean_val(c.get('main-group',''))
            subg = clean_val(c.get('subgroup',''))
            
            full_code = f"{s}{cl}{sub}{main}/{subg}".strip()
            if len(full_code) > 2:
                cpcs_set.add(full_code)

        resultados.append({
            'ID_Patente': f"{cc}{nr}{kc}",
            'Pais': cc, 'Num': nr, 'Tipo': kc,
            'Family_ID': doc.get('@family-id', ''),
            'Data_Publicacao': dt,
            'Requerente': requerentes_str,
            'Titulo': titulo,
            'CPCs': ", ".join(sorted(list(cpcs_set)))
        })

# ==========================================
# 3. LOGICA DE EXECUCAO PRINCIPAL
# ==========================================

try:
    # Divisao do dicionario de pesquisa em blocos menores
    tamanho_bloco = 4
    blocos_termos = [termos_rna[i:i + tamanho_bloco] for i in range(0, len(termos_rna), tamanho_bloco)]
    
    ids_totais = []
    for a in anos_para_pesquisar:
        print(f"\n[INFO] A processar o ano {a}. Dividindo os {len(termos_rna)} termos em {len(blocos_termos)} pesquisas parciais...")
        
        for idx, bloco in enumerate(blocos_termos):
            print(f"  -> A executar pesquisa parcial {idx + 1}/{len(blocos_termos)}...")
            
            txt_query = " OR ".join([f'ta="{termo}"' for termo in bloco])
            cql_parcial = f'({txt_query}) AND {cpc_query}'
            
            if FILTRAR_POR_APPLICANT and NOME_APPLICANT:
                cql_parcial = f'{cql_parcial} AND pa="{NOME_APPLICANT}"'
            
            ids_totais.extend(pesquisar_ids(cql_parcial, a))
            time.sleep(1)
            
    ids_totais = list(set(ids_totais))

    if not ids_totais:
        print("\n[AVISO] Nenhuma patente encontrada com os parametros fornecidos.")
    else:
        final_data = obter_detalhes(ids_totais)
        if final_data:
            df = pd.DataFrame(final_data)
            
            # ------------------------------------------
            # FILTROS DE EXCLUSAO AGRESSIVOS
            # ------------------------------------------
            prefixos_proibidos = ['A01H', 'A01N', 'A01P', 'C05', 'A61M', 'C12P', 'C12N5', 'C12N15/82', 'C12N15/80', 'C07K14/4356']
            padrao_cpc = '|'.join(prefixos_proibidos)
            df = df[~df['CPCs'].str.contains(padrao_cpc, case=False, na=False, regex=True)]
            
            df = df[~df['Titulo'].str.contains(termos_vet_agri, case=False, na=False, regex=True)]
            
            df = df.sort_values('Data_Publicacao').drop_duplicates('Family_ID', keep='first')

            df['Link_Espacenet'] = "https://worldwide.espacenet.com/publicationDetails/biblio?CC=" + \
                                   df['Pais'] + "&NR=" + df['Num'] + "&KC=" + df['Tipo'] + "&FT=D"
            
            # ------------------------------------------
            # SINALIZACAO DE CURADORIA
            # ------------------------------------------
            df['Aviso_Tecnologia'] = np.where(df['Titulo'].str.contains(termos_concorrentes, case=False, na=False, regex=True),
                                              'VERIFICAR (Possivel miRNA/CRISPR/ASO)', '')
            
            df['Curacao_Manual'] = ""
            df['Notas'] = ""

            if FILTRAR_POR_APPLICANT:
                nome_ficheiro = f'dataset_siRNA_{NOME_APPLICANT}.csv'
            elif len(anos_para_pesquisar) == 1:
                nome_ficheiro = f'dataset_siRNA_{anos_para_pesquisar[0]}.csv'
            else:
                nome_ficheiro = 'dataset_siRNA_run.csv'
            
            colunas = ['ID_Patente', 'Data_Publicacao', 'Requerente', 'Titulo', 'Link_Espacenet', 'Aviso_Tecnologia', 'Curacao_Manual', 'Notas', 'CPCs', 'Family_ID']
            df[colunas].to_csv(nome_ficheiro, index=False, sep=';', encoding='utf-8-sig')

            print(f"\n[SUCESSO] Processamento concluido. Exportadas {len(df)} familias de patentes unicas para o ficheiro {nome_ficheiro}.")
            
            # ==========================================
            # 4. AUDITORIA E RELATORIO DE QUOTA FINAL
            # ==========================================
            print("\n[INFO] A obter relatorio de consumo de quota da API...")
            tk_final = obter_token_valido()
            url_quota = "https://ops.epo.org/3.2/rest-services/published-data/search?q=ti=siRNA"
            headers_quota = {'Authorization': f'Bearer {tk_final}', 'Accept': 'application/json', 'X-OPS-Range': '1-1'}
            res_quota = requests.get(url_quota, headers=headers_quota)
            
            if res_quota.status_code == 200:
                usado = int(res_quota.headers.get('X-RegisteredQuotaPerWeek-Used', 0))
                limite = int(res_quota.headers.get('X-RegisteredQuotaPerWeek-Limit', 4294967296))
                usado_mb = usado / (1024 * 1024)
                limite_gb = limite / (1024 * 1024 * 1024)
                percentagem = (usado / limite) * 100 if limite > 0 else 0
                
                print("\n" + "="*40)
                print(" ESTADO DA QUOTA EPO (SEMANAL)")
                print("="*40)
                print(f"Limite Semanal : {limite_gb:.2f} GB")
                print(f"Dados Usados   : {usado_mb:.2f} MB ({percentagem:.4f}%)")
                print("="*40)

except Exception as e:
    print(f"\n[ERRO CRITICO] A execucao falhou devido a uma excepcao no sistema: {e}")


[INFO] A processar o ano 2026. Dividindo os 29 termos em 8 pesquisas parciais...
  -> A executar pesquisa parcial 1/8...

[SEGURANCA] A gerar um novo Token de Acesso EPO...
  -> A executar pesquisa parcial 2/8...
  -> A executar pesquisa parcial 3/8...
  -> A executar pesquisa parcial 4/8...
  -> A executar pesquisa parcial 5/8...
  -> A executar pesquisa parcial 6/8...
  -> A executar pesquisa parcial 7/8...
  -> A executar pesquisa parcial 8/8...

[INFO] A iniciar extracao de metadados para 380 patentes...

[SUCESSO] Processamento concluido. Exportadas 289 familias de patentes unicas para o ficheiro dataset_siRNA_2026.csv.

[INFO] A obter relatorio de consumo de quota da API...

 ESTADO DA QUOTA EPO (SEMANAL)
Limite Semanal : 4.00 GB
Dados Usados   : 289.00 MB (7.0556%)


=============================================================================
### Pipeline de Processamento e Harmonização de Dados de Patentes (siRNA: 2026)
=============================================================================

Este módulo constitui a segunda fase da pipeline, focada no processamento de linguagem natural (NLP) básico e na limpeza de dados bibliográficos. O objetivo primário é a normalização das entidades (Requerentes/Applicants), garantindo que variações lexicais, abreviaturas ou erros de preenchimento não enviesem a análise estatística subsequente.

O script transforma dados brutos extraídos da API OPS-EPO num dataset consolidado e pronto para importação e visualização avançada em ambiente RStudio.

### Funcionalidades de Processamento:
1. Consolidação e Harmonização de Entidades:
Implementação de um algoritmo de unificação baseado em dicionários de equivalência. Este processo agrupa variações de nomes (ex: "PEKING UNIVERSITY" e "BEIJING UNIVERSITY") numa única entidade padrão, permitindo cálculos precisos de market share tecnológico.

2. Normalização de Caracteres e Remoção de Diacríticos:
Aplicação de técnicas de normalização Unicode para remover acentos e caracteres especiais (ex: "LIÈGE" torna-se "LIEGE"). Esta etapa elimina falsos positivos causados por diferenças de codificação regional.

3. Purga de Inventores Individuais:
Aplicação de uma lista de exclusão (Blacklist) para filtrar nomes de pessoas físicas que constam erradamente como requerentes principais, garantindo que o dataset final represente apenas instituições de investigação e empresas.

4. Padronização de Sufixos Jurídicos:
Utilização de expressões regulares (RegEx) para remover sufixos comerciais e jurídicos (ex: INC, LTD, GMBH, LLC), focando a análise no nome core das organizações.

5. Tratamento de Inversões de Nomenclatura:
Deteção automática e correção de inversões comuns em traduções de universidades asiáticas (ex: "UNIVERSITY XIAMEN" corrigido para "XIAMEN UNIVERSITY").

6. Exportação Otimizada para RStudio:
Geração de um ficheiro CSV com codificação standard e separadores compatíveis com as bibliotecas readr e ggplot2, facilitando a criação imediata de gráficos de barras, redes de colaboração ou séries temporais.

=============================================================================

In [66]:
def remover_acentos(texto):
    """
    Remove acentos e caracteres especiais.
    Transforma 'È' em 'E', 'Á' em 'A', etc.
    Fundamental para que o computador reconheça nomes iguais escritos de forma diferente.
    """
    texto = unicodedata.normalize('NFD', texto)
    texto = texto.encode('ascii', 'ignore').decode("utf-8")
    return str(texto)

def calcular_top_applicants(caminho_ficheiro, top_n=100):
    print(f"[INFO] A carregar e processar dados do ficheiro: {caminho_ficheiro}...")
    
    # 1. Leitura do ficheiro gerado na extracao inicial
    df = pd.read_csv(caminho_ficheiro, sep=';')
    registos_expandidos = []
    
    # ---------------------------------------------------------
    # 2. LISTA DE EXCLUSAO: Inventores
    # Nomes de investigadores que nao devem ser contabilizados como empresas.
    # ---------------------------------------------------------
    inventores_ignorar = [
        "IWAMOTO NAOKI", "LIU WEI", "LUU KHOA", "MARAPPAN", 
        "VARGEESE", "OWEN ADRIANA", "YANG HSIU", "LAMATTINA", 
        "BYRNE", "SINGH KULDEEP", "SHAH HIMALI", "KAWAMOTO",
        "GHOSH", "HAEGELE", "HU YANBIN", "HE HUIJUN", "LU JIANYU", 
        "HE HAIYING", "CHEN SHUHUI", "GLEBOCKA", "KLOSSOWSKI", "SHUVAEV",
        "DESAI JIGAR", "PRAKASHA PRIYANKA", "LONGO KENNETH", "DESAI", "LONGO", "PRAKASHA",
        "PEI TAO", "BLOKHIN ANDREI", "CHEN JING", "ENDEAN THOMAS", "BENSON JONATHAN",
        "KANDASAMY", "LU GENLIANG", "KUMARASAMY", "CHATTERJEE"
    ]
    
    # ---------------------------------------------------------
    # 3. DICIONARIO DE CONSOLIDACAO
    # Define o nome padrao para cada grupo de variacoes detetado.
    # ---------------------------------------------------------
    dicionario_unificacao = {
        # Agrupamento de Universidades
        "WISCONSIN": "MEDICAL COLLEGE OF WISCONSIN",
        "MASSACHUSETTS": "UNIVERSITY OF MASSACHUSETTS",
        "BRITISH COLUMBIA": "UNIVERSITY OF BRITISH COLUMBIA",
        "PEKING": "PEKING UNIVERSITY",
        "BEIJING": "PEKING UNIVERSITY", 
        "ANTWERP": "UNIVERSITY OF ANTWERP",
        "LIEGE": "UNIVERSITY OF LIEGE",
        "VIRGINIA": "UNIVERSITY OF VIRGINIA",
        "TEXAS": "UNIVERSITY OF TEXAS",
        "TORONTO": "UNIVERSITY OF TORONTO",
        "CALIFORNIA": "UNIVERSITY OF CALIFORNIA",
        "FLORIDA": "UNIVERSITY OF FLORIDA",
        "JOHNS HOPKINS": "JOHNS HOPKINS UNIVERSITY",
        "TOKYO MEDICAL": "TOKYO MEDICAL UNIVERSITY",
        "KOBE": "KOBE UNIVERSITY",
        "DALIAN MINZU": "DALIAN MINZU UNIVERSITY",
        "XUZHOU": "XUZHOU MEDICAL UNIVERSITY",
        "DANA FARBER": "DANA FARBER CANCER INSTITUTE",
        "KOREA INST": "KOREA INSTITUTE OF SCIENCE AND TECHNOLOGY",
        "SUNGKYUNKWAN": "SUNGKYUNKWAN UNIVERSITY",
        "DELAWARE": "DELAWARE STATE UNIVERSITY",
        "DELFT": "DELFT UNIVERSITY OF TECHNOLOGY",
        "ULSAN": "ULSAN NATIONAL INSTITUTE OF SCIENCE AND TECHNOLOGY",
        "TOKAI": "TOKAI NATIONAL HIGHER EDUCATION AND RESEARCH SYSTEM",
        
        # Correcao de inversoes de nomes
        "UNIVERSITY NANTONG": "NANTONG UNIVERSITY",
        "UNIVERSITY NORTHWESTERN POLYTECHNICAL": "NORTHWESTERN POLYTECHNICAL UNIVERSITY",
        "UNIVERSITY XIAMEN": "XIAMEN UNIVERSITY",
        "UNIVERSITY HEBEI TECHNOLOGY": "HEBEI UNIVERSITY OF TECHNOLOGY",
        "UNIVERSITY EAST CHINA NORMAL": "EAST CHINA NORMAL UNIVERSITY",
        "UNIVERSITY NANJING AGRICULTURAL": "NANJING AGRICULTURAL UNIVERSITY",
        "UNIVERSITY SOUTH CHINA AGRICULT": "SOUTH CHINA AGRICULTURAL UNIVERSITY",
        "UNIVERSITY KYOTO": "KYOTO UNIVERSITY",

        # Institutos e Unidades Hospitalares
        "CARDIOLOGIE DE MONTREAL": "MONTREAL HEART INSTITUTE",
        "NATIONWIDE CHILDREN": "NATIONWIDE CHILDRENS HOSPITAL",
        "CHILDRENS MEDICAL": "CHILDRENS HOSPITAL MEDICAL CENTER",
        "CHILDRENS HOSPITAL MED": "CHILDRENS HOSPITAL MEDICAL CENTER",
        "GENOME RES": "GENOME RESEARCH",
        "CMS RES": "CMS RESEARCH & DEVELOPMENT",
        
        # Empresas e Farmaceuticas
        "LILLY": "ELI LILLY",
        "EXORNA": "EXORNA BIOSCIENCE",
        "RONA": "RONA THERAPEUTICS",
        "BISIRNA": "BISIRNA THERAPEUTICS",
        "JENKEM": "JENKEM TECHNOLOGY",
        "QILU": "QILU PHARMACEUTICAL",
        "ANSHUN": "SHANDONG ANSHUN PHARMACEUTICAL",
        "CHIA TAI": "CHIA TAI TIANQING PHARMACEUTICAL GROUP",
        "ALNYLAM": "ALNYLAM PHARMACEUTICALS",
        "ARROWHEAD": "ARROWHEAD PHARMACEUTICALS",
        "SUZHOU RIBO": "SUZHOU RIBO LIFE SCIENCE",
        "RIBO LIFE SCIENCE": "SUZHOU RIBO LIFE SCIENCE",
        "DYNE": "DYNE THERAPEUTICS",
        "TUOJIE": "TUOJIE BIOTECH SHANGHAI",
        "BEBETTER": "BEBETTER MED",
        "YINGTE": "YINGTE MED TECH",
        "TAKEDA": "TAKEDA PHARMACEUTICALS",
        "JANSSEN": "JANSSEN PHARMACEUTICALS",
        "MPEG LA": "MPEG LA",
        "SRINA": "SIRNA THERAPEUTICS", 
        "SIRNA THERAPEUTICS": "SIRNA THERAPEUTICS",
        "REGENERON": "REGENERON PHARMACEUTICALS",
        "HANSOH": "HANSOH PHARMACEUTICAL",
        "NOVO NORDISK": "NOVO NORDISK",
        "OLIX": "OLIX PHARMACEUTICALS",
        "FRANCAISE CONTRE LES MYOPATHIES": "ASSOCIATION FRANCAISE CONTRE LES MYOPATHIES",
        "HUMANWELL": "HUMANWELL HEALTHCARE GROUP"
    }
    
    # 4. Processamento de limpeza de cada linha da tabela
    for index, row in df.iterrows():
        family_id = row['Family_ID']
        
        if pd.isna(row['Requerente']):
            continue
            
        requerentes_brutos = str(row['Requerente']).split('|')
        aplicantes_unicos_nesta_patente = set()
        
        for req in requerentes_brutos:
            # Converter para maiusculas e remover codigos de pais entre parentesis
            req = req.strip().upper()
            req = re.sub(r'\[.*?\]', '', req) 
            
            # Normalizacao de caracteres e remocao de pontuacao
            req = remover_acentos(req)
            req = req.replace(',', '').replace('.', '').replace('(', '').replace(')', '').replace('-', ' ').replace("'", "").replace("&", "AND")
            
            # Remover o artigo "THE" se estiver no inicio do nome
            req = re.sub(r'^THE\s+', '', req)
            
            # Padronizar a abreviatura de Universidade
            req = re.sub(r'\bUNIV\b', 'UNIVERSITY', req)
            
            # Remover sufixos juridicos/comerciais (INC, LTD, CORP, etc.)
            sufixos = r'\b(INC|INCORPORATED|CORP|CORPORATION|L L C|LLC|LTD|LIMITED|CO|COMPANY|GMBH|SA|PLC|NV|A/S|AG|PTY|PTE|AND)\b'
            req = re.sub(sufixos, '', req)
            req = re.sub(r'\s+', ' ', req).strip() # Eliminar espacos duplos
            
            # Validar se o nome resultante e valido
            if not req or not re.search('[A-Z]', req):
                continue
                
            # Validar contra a lista de inventores (exclusao)
            e_inventor = False
            for inventor in inventores_ignorar:
                if inventor in req:
                    e_inventor = True
                    break
            if e_inventor:
                continue
            
            # Aplicar a consolidacao via dicionario
            for chave, valor in dicionario_unificacao.items():
                if chave in req:
                    req = valor
                    break
            
            aplicantes_unicos_nesta_patente.add(req)
            
        for app in aplicantes_unicos_nesta_patente:
            registos_expandidos.append({'Family_ID': family_id, 'Empresa': app})
            
    # 5. Agregacao dos resultados consolidados
    df_exp = pd.DataFrame(registos_expandidos)
    
    # Contagem de familias de patentes por requerente
    top_applicants = df_exp['Empresa'].value_counts().reset_index()
    top_applicants.columns = ['Empresa', 'Numero_Patentes'] 
    
    # Selecao do Top N para o relatorio
    df_top = top_applicants.head(top_n)
    
    print("\n" + "="*50)
    print(f" TOP {top_n} APPLICANTS PROCESSADOS ")
    print("="*50)
    
    # 6. Geracao do ficheiro para RStudio
    nome_ficheiro_r = f'top{top_n}_{caminho_ficheiro}.csv'
    df_top.to_csv(nome_ficheiro_r, index=False, sep=',')
    print(f"[SUCESSO] Ficheiro '{nome_ficheiro_r}' pronto para analise\n")
    
    return df_top

In [67]:
# ==========================================
# EXECUCAO
# ==========================================
df_estatistica = calcular_top_applicants('dataset_siRNA_2026.csv', top_n=100)

display(df_estatistica)

[INFO] A carregar e processar dados do ficheiro: dataset_siRNA_2026.csv...

 TOP 100 APPLICANTS PROCESSADOS 
[SUCESSO] Ficheiro 'top100_dataset_siRNA_2026.csv.csv' pronto para analise



,Empresa,Numero_Patentes
0,ALNYLAM PHARMACEUTICALS,32
1,ARROWHEAD PHARMACEUTICALS,24
2,PEKING UNIVERSITY,20
3,SUZHOU RIBO LIFE SCIENCE,10
4,SHANGHAI ARGO BIOPHARMACEUTICAL,8
...,...,...
95,VERSITI BLOOD RESEARCH INSTITUTE FOUNDATION,1
96,NANJING DRUM TOWER HOSPITAL,1
97,SHENZHEN SALUBRIS PHARM,1
98,UNIVERSITY OF VIRGINIA,1
